In [2]:
"""
model.py
---------
This module defines the mathematical model for the PEM electrolyzer membrane design.
It provides objective functions and a constraint based on mechanical durability.
"""

import numpy as np

# Constants for efficiency calculation
F = 96485.0  # Faraday constant (C/mol)
HHV_H2 = 286000.0  # Higher Heating Value of H2 (J/mol)
V0 = 1.23  # Base cell voltage (V)
sigma = 10.0  # Ionic conductivity (S/m)

# Conversion: current density from A/cm² to A/m².
def convert_j(j):
    return j * 1e4

# Energy Efficiency: Using a simplified model
def efficiency(t, j):
    # Convert j to A/m².
    j_m = convert_j(j)
    # Model the cell voltage as a sum of a base voltage and ohmic losses:
    V_cell = V0 + (j_m * t) / sigma
    # Effective voltage per mole of H2 produced (via Faraday's law, n = 2).
    HHV_eff = HHV_H2 / (2 * F)
    eta = HHV_eff / V_cell
    return eta

def objective1(x):
    # x[0] = t (membrane thickness in m), x[1] = j (current density in A/cm²)
    t = x[0]
    j = x[1]
    eta = efficiency(t, j)
    # Since we maximize efficiency, we minimize its negative.
    return -eta

# Durability: A simple exponential decay model based on j and t.
L0 = 100000.0  # Baseline lifetime (hours)
gamma = 1e-6   # Degradation parameter (arbitrary)
def lifetime(t, j):
    j_m = convert_j(j)
    # Lifetime decreases as j increases and as t decreases.
    L = L0 * np.exp(-gamma * (j_m / t))
    return L

def objective2(x):
    t = x[0]
    j = x[1]
    L = lifetime(t, j)
    return -L  # Maximizing lifetime → minimize (-L)

# Cost: Consider only membrane material and manufacturing cost.
c_ionomer = 300.0    # Cost per kg of membrane material ($/kg)
density = 2000.0     # Material density (kg/m³)
c_manuf = 10.0       # Manufacturing cost per m² ($/m²)
def cost(t):
    # Material cost per unit area = c_ionomer * (density * t)
    return c_ionomer * density * t + c_manuf

def objective3(x):
    t = x[0]
    return cost(t)

# Environmental Impact: Proportional to the mass of membrane used.
c_E = 5.0   # Environmental impact factor (kg CO₂-eq per kg)
def environmental_impact(t):
    return c_E * density * t

def objective4(x):
    t = x[0]
    return environmental_impact(t)

# Mechanical durability constraint:
# We require t >= t_min_mech, where:
#   t_min_mech = r * sqrt( (k * ΔP * SF) / σ_tensile )
r = 0.005         # Effective radius (m); e.g., for a 1-cm diameter cell, r ≈ 0.005 m.
k = 1.0           # Geometric constant (dimensionless)
DeltaP = 1e5      # Pressure difference (Pa)
SF = 2.0          # Safety factor
sigma_tensile = 100e6  # Tensile strength (Pa)
def t_min_mech():
    return r * np.sqrt((k * DeltaP * SF) / sigma_tensile)

def constraint(x):
    t = x[0]
    # Constraint is satisfied if: t - t_min_mech() >= 0.
    return t - t_min_mech()

# Evaluate all objectives and constraint for a set of design points.
def evaluate_objectives(X):
    """
    Parameters:
      X : numpy array of shape (n_individuals, n_variables)
          Each row is a design vector [t, j].
    
    Returns:
      F : numpy array of shape (n_individuals, 4)
          Objective values for each design point.
      G : numpy array of shape (n_individuals,)
          Constraint values (feasible if >= 0).
    """
    n = X.shape[0]
    f1 = np.zeros(n)
    f2 = np.zeros(n)
    f3 = np.zeros(n)
    f4 = np.zeros(n)
    g = np.zeros(n)
    
    for i in range(n):
        xi = X[i, :]
        t = xi[0]
        j = xi[1]
        f1[i] = objective1(xi)
        f2[i] = objective2(xi)
        f3[i] = objective3(xi)
        f4[i] = objective4(xi)
        g[i] = constraint(xi)
    
    F = np.column_stack([f1, f2, f3, f4])
    return F, g

if __name__ == "__main__":
    # Quick test: Print the minimum required thickness and evaluate a sample design.
    print("Minimum mechanical thickness (t_min_mech):", t_min_mech(), "m")
    x_sample = np.array([t_min_mech() + 50e-6, 1.0])  # sample: thickness 50 microns above minimum, j=1.0 A/cm²
    F_sample, g_sample = evaluate_objectives(np.array([x_sample]))
    print("Sample objectives:", F_sample)
    print("Sample constraint value (should be >= 0):", g_sample)


Minimum mechanical thickness (t_min_mech): 0.00022360679774997898 m
Sample objectives: [[-9.85693643e-01 -1.33985575e-11  1.74164079e+02  2.73606798e+00]]
Sample constraint value (should be >= 0): [5.e-05]


In [ ]:
"""
optimization.py
----------------
This module sets up and solves the multiobjective optimization problem for the PEM electrolyzer membrane design
using the pymoo library. We use NSGA2 (a Pareto-based method) for demonstration.
"""

import numpy as np
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.termination import get_sampling, get_crossover, get_mutation
from pymoo.optimize import minimize
import matplotlib.pyplot as plt

# from model import evaluate_objectives, t_min_mech

# Define the custom multiobjective problem for PEM membrane design.
class PEMProblem(Problem):
    def __init__(self):
        # Decision vector: x = [t, j]
        # t (membrane thickness): lower bound = max(t_lower_bound, t_min_mech), upper bound = 300e-6 m.
        # Here, we set t_lower_bound = t_min_mech (as computed) for feasibility.
        t_lower = t_min_mech()   # in m
        t_upper = 300e-6         # 300 microns
        # Current density j in A/cm²: bounds [0.5, 2.0]
        xl = np.array([t_lower, 0.5])
        xu = np.array([t_upper, 2.0])
        super().__init__(n_var=2,
                         n_obj=4,
                         n_constr=1,
                         xl=xl,
                         xu=xu)
    
    def _evaluate(self, X, out, *args, **kwargs):
        # Evaluate objectives and constraints.
        F, G = evaluate_objectives(X)
        out["F"] = F
        # pymoo uses constraints as feasible when G <= 0.
        # Our constraint function is defined as: t - t_min_mech() >= 0.
        # To use pymoo's convention, we set:
        out["G"] = -G  # Feasible if -G <= 0 (i.e. if G >= 0)

if __name__ == '__main__':
    # Instantiate the problem.
    problem = PEMProblem()
    
    # Define NSGA2 algorithm settings.
    algorithm = NSGA2(
        pop_size=100,
        sampling=get_sampling("real_random"),
        crossover=get_crossover("real_sbx", prob=0.9, eta=15),
        mutation=get_mutation("real_pm", eta=20),
        eliminate_duplicates=True
    )
    
    # Run the optimization for 100 generations.
    res = minimize(problem,
                   algorithm,
                   ('n_gen', 100),
                   seed=1,
                   verbose=True)
    
    # Print the results.
    print("Best solutions (decision variables) found:")
    print(res.X)
    print("Objective values:")
    print(res.F)
    
    # Plot a Pareto front for a pair of objectives: e.g., f1 (negative efficiency) vs. f3 (cost).
    plt.figure()
    plt.scatter(res.F[:, 0], res.F[:, 2], c='blue', marker='o')
    plt.xlabel("Objective 1: -Efficiency")
    plt.ylabel("Objective 3: Cost")
    plt.title("Pareto Front (Objective 1 vs. Objective 3)")
    plt.grid(True)
    plt.show()


ModuleNotFoundError: No module named 'pymoo.factory'